# Week 8 — Multi-Layer Perceptron (MLP)
So far, scikit-learn has hidden the training loop from us (`.fit()` did
everything). This week we open that black box: PyTorch makes you write the
loop yourself, which is exactly what lets you build custom architectures later

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Project: Seoul Bike Sharing Demand Prediction

**주제:** 서울시 공유 자전거 대여량 예측 (Seoul Bike Sharing Demand Prediction)
**문제 유형:** Regression
**모델:** Multi-Layer Perceptron (MLP), PyTorch
**데이터셋:** [UCI Seoul Bike Sharing Demand Dataset](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)
(날씨/시간/계절 정보로 2017-2018년 서울시 시간당 공유 자전거 대여량을 기록한 실제 데이터 — 8,760행)

**정형(tabular) 데이터**로 **회귀(regression)** 문제를 풀어봅니다 — 날씨와 시간 정보로
"오늘 이 시간에 자전거가 몇 대나 대여될까?"를 예측하는 실제 도시 데이터 문제입니다.

이 프로젝트를 통해 다음을 직접 경험합니다:
1. 정형 데이터(tabular data)를 이용한 회귀 문제
2. 데이터 전처리 및 feature scaling의 필요성 (그리고 data leakage를 막는 방법)
3. PyTorch로 MLP를 직접 구현하기
4. Loss function과 optimizer로 모델을 학습시키기
5. **hidden layer의 neuron 개수**가 모델 성능에 미치는 영향

### 1. 데이터 불러오기

UCI 공식 페이지(`archive.ics.uci.edu`)는 환경에 따라 직접 접속이 막혀 있을 수 있어,
동일한 내용을 그대로 담고 있는 GitHub 미러에서 내려받습니다. 인코딩이 일반적인
UTF-8이 아니라 `cp1252`라는 점에 주의하세요 (`°C` 같은 특수문자가 깨지는 걸 방지).

In [ ]:
import urllib.request

BIKE_DATA_URL = "https://raw.githubusercontent.com/yahya-ansariii/BikeSharingDemandRegression/master/SeoulBikeData.csv"
urllib.request.urlretrieve(BIKE_DATA_URL, "SeoulBikeData.csv")

bike_df = pd.read_csv("SeoulBikeData.csv", encoding="cp1252")
print("Shape:", bike_df.shape)
print(bike_df.dtypes)
print(
    f"\nRented Bike Count range: {bike_df['Rented Bike Count'].min()} ~ "
    f"{bike_df['Rented Bike Count'].max()} (mean={bike_df['Rented Bike Count'].mean():.1f}) "
    "-- RMSE 값을 볼 때 이 범위 대비 얼마나 작은지로 체감해보세요."
)
bike_df.head()

### 2. 불필요한 데이터 확인 및 제거

먼저 결측치가 있는지 확인합니다. 그리고 `Date` 컬럼은 문자열(날짜) 그대로는 모델
입력으로 쓸 수 없고, 이미 `Hour`(시간)와 `Seasons`(계절) 컬럼이 날짜가 담고 있는
핵심 정보(시간대, 계절성)를 대신 제공하므로 **입력 변수 목록에서 제외**합니다.

한 가지 더 체크할 점: `Functioning Day`(대여소 운영 여부)가 `No`인 시간대는 대여량이
항상 0입니다 — 이 변수는 실습 요구사항의 입력 변수 목록에 포함되어 있으므로, 이 행들을
삭제하지 않고 그대로 남겨둡니다. 모델이 "운영을 안 하면 대여량은 0"이라는 규칙까지
스스로 학습하게 됩니다.

In [ ]:
print("결측치 개수:")
print(bike_df.isna().sum().sum())

print("\nFunctioning Day == 'No'인 행의 대여량 통계:")
print(bike_df.loc[bike_df["Functioning Day"] == "No", "Rented Bike Count"].describe())

bike_df = bike_df.drop(columns=["Date"])
print("\nDate 컬럼 제거 후 shape:", bike_df.shape)

### 3. 범주형 변수 처리

- `Holiday`(공휴일 여부), `Functioning Day`(운영 여부)는 두 값(Yes/No류)만 가지는
  **이진 변수**라 0/1로 바로 바꿉니다.
- `Seasons`(계절)는 네 값 사이에 순서가 없는 **명목(nominal) 변수**라
  `pd.get_dummies`로 **원-핫 인코딩**합니다.

In [ ]:
bike_df["Holiday"] = (bike_df["Holiday"] == "Holiday").astype(int)
bike_df["Functioning Day"] = (bike_df["Functioning Day"] == "Yes").astype(int)
bike_df = pd.get_dummies(bike_df, columns=["Seasons"])

print("인코딩 후 컬럼:", bike_df.columns.tolist())
bike_df.head()

### 4. 입력(X)/타겟(y) 분리, Train/Validation/Test 분리

타겟은 `Rented Bike Count`(시간당 대여량), 나머지가 입력 변수입니다. 데이터를
**학습(60%) / 검증(20%) / 테스트(20%)**로 나눕니다 — 검증셋은 학습 중 모델을
점검하는 용도, 테스트셋은 맨 마지막에 딱 한 번만 사용합니다.

In [ ]:
from sklearn.model_selection import train_test_split

bike_y = bike_df["Rented Bike Count"].values.astype("float32")
bike_X = bike_df.drop(columns=["Rented Bike Count"]).values.astype("float32")
bike_feature_names = bike_df.drop(columns=["Rented Bike Count"]).columns.tolist()
print(f"입력 변수 {len(bike_feature_names)}개:", bike_feature_names)

bike_X_trainval, bike_X_test, bike_y_trainval, bike_y_test = train_test_split(
    bike_X, bike_y, test_size=0.2, random_state=42
)
bike_X_train, bike_X_val, bike_y_train, bike_y_val = train_test_split(
    bike_X_trainval, bike_y_trainval, test_size=0.25, random_state=42
)
print(f"train/val/test 크기: {len(bike_X_train)} / {len(bike_X_val)} / {len(bike_X_test)}")

### 5. Feature Scaling (Data Leakage 방지)

입력 변수들은 범위가 서로 크게 다릅니다 (`Hour`는 0~23, `Visibility`는 수백~2000).
`StandardScaler`로 평균 0, 표준편차 1로 맞춰줍니다.

**중요:** `fit()`은 **학습 데이터에만** 적용하고, 검증/테스트 데이터에는 `transform()`만
적용합니다. 검증/테스트 데이터의 정보가 scaler에 스며들면(= data leakage) "한 번도 본
적 없는 데이터"라는 전제가 깨집니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

bike_scaler = StandardScaler().fit(bike_X_train)
bike_X_train_s = bike_scaler.transform(bike_X_train)
bike_X_val_s = bike_scaler.transform(bike_X_val)
bike_X_test_s = bike_scaler.transform(bike_X_test)

print("스케일링 후 train 평균 (0에 가까워야 함):", bike_X_train_s.mean().round(4))
print("스케일링 후 train 표준편차 (1에 가까워야 함):", bike_X_train_s.std().round(4))

### 6. PyTorch Tensor 변환 및 DataLoader 구성

학습 데이터는 `DataLoader`로 미니배치 단위로 꺼내 쓰고, 검증/테스트 데이터는 크기가
작으니 한 번에 통째로 모델에 넣습니다. 재현 가능한 실험을 위해 random seed를
고정하고, GPU가 있으면 자동으로 사용합니다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 device:", device)

bike_train_ds = TensorDataset(
    torch.tensor(bike_X_train_s, dtype=torch.float32),
    torch.tensor(bike_y_train, dtype=torch.float32).view(-1, 1),
)
bike_train_loader = DataLoader(bike_train_ds, batch_size=64, shuffle=True)

bike_X_val_t = torch.tensor(bike_X_val_s, dtype=torch.float32).to(device)
bike_y_val_t = torch.tensor(bike_y_val, dtype=torch.float32).view(-1, 1).to(device)
bike_X_test_t = torch.tensor(bike_X_test_s, dtype=torch.float32).to(device)
bike_y_test_t = torch.tensor(bike_y_test, dtype=torch.float32).view(-1, 1).to(device)

### 7. MLP 모델 구현

구조는 **Input → Hidden 1 → Hidden 2 → Output**으로 고정합니다:
- Hidden layer는 기본 **2개**, 두 layer의 neuron 개수는 **서로 같게** 둡니다
  (바꿀 수 있는 건 neuron 개수 하나뿐 — 이게 이번 실습의 핵심 실험 변수입니다). 뒤에서는
  hidden layer가 **1개뿐인 경우**도 design space에 추가해서, '더 넓게'와 '더 깊게'가
  같은 효과를 내는지도 비교해봅니다.
- 활성함수는 **ReLU**.
- **회귀** 문제라 Output layer에는 활성함수를 적용하지 않습니다 (범위 제한 없는
  실수값을 그대로 출력).
- Input/Output 크기는 데이터에 맞춰 자동으로 결정됩니다 (`input_dim`, 출력은 1개).

In [ ]:
class BikeDemandMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_hidden_layers=2):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.hidden_layers = nn.ModuleList(layers)
        self.output_layer = nn.Linear(in_dim, 1)  # 회귀 문제이므로 출력에 활성함수를 적용하지 않음

    def forward(self, x):
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        return self.output_layer(x)


example_model = BikeDemandMLP(input_dim=bike_X_train_s.shape[1], hidden_dim=16)
print(example_model)
print("총 파라미터 개수:", sum(p.numel() for p in example_model.parameters()))

### 8. 학습 함수 구현

모든 모델에 **완전히 동일한 학습 조건**(epoch 수, learning rate, batch size, optimizer,
loss function)을 적용해야 neuron 개수의 효과만 비교할 수 있습니다. 매 epoch마다
train loss와 validation loss를 모두 기록해서, 나중에 과적합(overfitting) 여부를
직접 확인할 수 있게 합니다.

학습의 네 단계(이 실습에서 PyTorch로 직접 작성하는 부분입니다):
**forward propagation → loss 계산 → backpropagation → optimizer step**

In [ ]:
N_EPOCHS = 500
LEARNING_RATE = 1e-3


def train_bike_model(hidden_dim, n_hidden_layers=2, seed=SEED):
    torch.manual_seed(seed)
    model = BikeDemandMLP(
        input_dim=bike_X_train_s.shape[1], hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    for epoch in range(N_EPOCHS):
        model.train()
        running_loss = 0.0
        for xb, yb in bike_train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()           # 이전 epoch의 gradient 초기화
            pred = model(xb)                # 1. forward propagation
            loss = criterion(pred, yb)      # 2. loss 계산 (MSE)
            loss.backward()                 # 3. backpropagation (gradient 계산)
            optimizer.step()                # 4. optimizer step (가중치 업데이트)

            running_loss += loss.item() * len(xb)
        train_losses.append(running_loss / len(bike_X_train_s))

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(bike_X_val_t), bike_y_val_t).item()
        val_losses.append(val_loss)

    return model, train_losses, val_losses

### 9. 다섯 가지 모델 학습: Model A (16) ~ Model D (128), 그리고 Model E (1개 layer)

Hidden layer 개수는 2개로 고정하고 neuron 개수만 16 → 32 → 64 → 128로 바꿔가며
**똑같은 조건**으로 학습시킵니다. 추가로, Model C와 neuron 개수(64)는 같지만
hidden layer가 **1개뿐인 Model E**도 함께 학습시켜, 같은 폭(width)에서 깊이(depth)만
줄였을 때 어떤 차이가 나는지 비교합니다. 5개 모델을 모두 학습하는 데 몇 분 정도
걸릴 수 있습니다.

In [ ]:
bike_model_configs = {
    "A": {"hidden_dim": 16, "n_hidden_layers": 2},
    "B": {"hidden_dim": 32, "n_hidden_layers": 2},
    "C": {"hidden_dim": 64, "n_hidden_layers": 2},
    "D": {"hidden_dim": 128, "n_hidden_layers": 2},
    "E": {"hidden_dim": 64, "n_hidden_layers": 1},  # Model C와 neuron 수는 같지만 layer가 1개뿐
}

bike_models, bike_histories = {}, {}
for name, config in bike_model_configs.items():
    model, train_losses, val_losses = train_bike_model(config["hidden_dim"], config["n_hidden_layers"])
    bike_models[name] = model
    bike_histories[name] = (train_losses, val_losses)
    print(f"Model {name} (hidden={config['hidden_dim']:3d}, layers={config['n_hidden_layers']}) 학습 완료 — "
          f"최종 train loss={train_losses[-1]:.1f}, val loss={val_losses[-1]:.1f}")

### 10. 테스트셋 성능 평가

학습에 전혀 사용하지 않은 테스트셋으로 **MAE, MSE, RMSE, R²**를 계산해서
모델별로 비교합니다. 함께, neuron 개수가 늘어날 때 학습 가능한 **parameter 개수**가
어떻게 늘어나는지도 같이 봅니다.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

bike_results = {}
for name, model in bike_models.items():
    model.eval()
    with torch.no_grad():
        test_pred = model(bike_X_test_t).cpu().numpy().ravel()

    n_params = sum(p.numel() for p in model.parameters())
    mae = mean_absolute_error(bike_y_test, test_pred)
    mse = mean_squared_error(bike_y_test, test_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(bike_y_test, test_pred)
    train_loss, val_loss = bike_histories[name][0][-1], bike_histories[name][1][-1]

    bike_results[name] = {
        "hidden_neurons": bike_model_configs[name]["hidden_dim"],
        "n_hidden_layers": bike_model_configs[name]["n_hidden_layers"],
        "n_params": n_params,
        "final_train_loss": train_loss,
        "final_val_loss": val_loss,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
    }

bike_results_df = pd.DataFrame(bike_results).T
bike_results_df

### 11. 그래프로 결과 확인하기

세 가지를 그려봅니다:
1. 모델별 epoch에 따른 train/validation loss 곡선 — train loss와 val loss가
   벌어지는 모델이 있는지 (overfitting의 신호) 확인합니다.
2. neuron 개수에 따른 test RMSE — 성능이 꾸준히 좋아지는지, 어느 지점부터
   개선이 둔화되는지 확인합니다.
3. 시간대(Hour)별 실제 평균 대여량 vs. 모델의 평균 예측값 — 아침/저녁 통근 시간에
   솟구치는 뚜렷한 비선형 패턴을 모델이 얼마나 잘 따라가는지 봅니다 (MLP가 단순한
   직선이 아니라 복잡한 형태의 함수도 표현할 수 있다는 걸 가장 직접적으로 보여줍니다).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, (train_losses, val_losses)) in zip(axes.flat, bike_histories.items()):
    config = bike_model_configs[name]
    ax.plot(train_losses, label="train loss")
    ax.plot(val_losses, label="val loss")
    ax.set_title(f"Model {name} (hidden={config['hidden_dim']}, layers={config['n_hidden_layers']})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
for ax in axes.flat[len(bike_histories):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
two_layer = bike_results_df[bike_results_df["n_hidden_layers"] == 2].sort_values("hidden_neurons")
one_layer = bike_results_df[bike_results_df["n_hidden_layers"] == 1]

plt.figure(figsize=(6, 4))
plt.plot(two_layer["hidden_neurons"], two_layer["RMSE"], marker="o", label="2 hidden layers")
plt.scatter(one_layer["hidden_neurons"], one_layer["RMSE"], marker="x", s=100, color="red", label="1 hidden layer")
for name in bike_results_df.index:
    plt.annotate(name, (bike_results_df.loc[name, "hidden_neurons"], bike_results_df.loc[name, "RMSE"]))
plt.xlabel("Hidden Layer Neurons (per layer)")
plt.ylabel("Test RMSE")
plt.title("Test RMSE vs. Number of Hidden Neurons")
plt.legend()
plt.show()

In [ ]:
best_model_name = bike_results_df["RMSE"].astype(float).idxmin()
best_model = bike_models[best_model_name]
best_model.eval()
with torch.no_grad():
    best_pred = best_model(bike_X_test_t).cpu().numpy().ravel()

# MLP는 완전한 선형이 아니라 복잡한 형태의 함수도 표현할 수 있습니다 -- 이를 가장 잘 보여주는
# 입력은 Hour입니다: 실제 대여량은 "아침/저녁 통근 시간에 솟구치는" 뚜렷한 비선형 패턴을
# 보입니다. 테스트셋을 시간대별로 묶어서, 실제 평균 대여량과 모델의 평균 예측값을 같이 그려봅니다.
hour_idx = bike_feature_names.index("Hour")
test_hours = bike_X_test[:, hour_idx]
hourly_actual = pd.Series(bike_y_test).groupby(test_hours).mean()
hourly_pred = pd.Series(best_pred).groupby(test_hours).mean()

plt.figure(figsize=(8, 5))
plt.plot(hourly_actual.index, hourly_actual.values, marker="o", label="actual (test set average)")
plt.plot(hourly_pred.index, hourly_pred.values, marker="o", label=f"predicted (Model {best_model_name})")
plt.xlabel("Hour of Day")
plt.ylabel("Rented Bike Count")
plt.title("Actual vs. Predicted Demand by Hour of Day")
plt.legend()
plt.show()

### 12. 핵심 과제: Neuron 개수에 따른 성능 비교

| Model | Hidden Layer 1 | Hidden Layer 2 |
|---|---:|---:|
| A | 16 | 16 |
| B | 32 | 32 |
| C | 64 | 64 |
| D | 128 | 128 |
| E | 64 | (layer 없음, 1개짜리) |

위에서 나온 표와 그래프를 보고 다음 질문에 답해보세요.

1. Neuron 개수가 증가할수록 모델의 성능(RMSE, R²)은 어떻게 변하는가?
2. Neuron 개수가 증가하면 학습 가능한 parameter 수는 어떻게 변하는가? (표의
   `n_params` 열을 보고, 왜 이렇게 늘어나는지 — 특히 두 hidden layer 사이의
   연결이 어떤 역할을 하는지 — 생각해보세요.)
3. Neuron 개수가 많아지면 항상 좋은 성능을 얻을 수 있는가?
4. Training loss와 validation loss를 비교했을 때, overfitting이 발생하는(또는
   발생할 조짐이 보이는) 모델이 있는가? 몇 번 그래프에서 그 신호를 볼 수 있는가?
5. 가장 적절한 neuron 개수를 가진 모델은 무엇이며, 그렇게 판단한 이유는 무엇인가?
   (성능만이 아니라 parameter 개수, overfitting 정도까지 함께 고려해보세요.)

**참고 — 이게 전부는 아닙니다.** 이번 실습에서는 hidden layer 개수와 learning rate,
batch size, epoch 수를 전부 고정한 채 **neuron 개수 하나만** 바꿔봤습니다. 실제로는
이 모든 설정(learning rate, batch size, epoch 수, hidden layer 개수 자체 등)을 함께
탐색해야 최적의 모델을 찾을 수 있는데, 이 전체 과정을 **hyperparameter tuning**이라고
합니다. Week 6에서 XGBoost를 k-fold cross-validation으로 튜닝했던 것처럼, 여기서도
`GridSearchCV`나 `Optuna` 같은 도구로 여러 설정을 동시에 탐색할 수 있습니다 — 지금은
neuron 개수 하나만 바꿔도 결과가 얼마나 달라지는지를 먼저 체감하는 데 집중합니다.

**참고 — neuron 개수만이 다가 아닙니다.** Model E는 Model C와 neuron 개수(64)는
똑같지만 hidden layer가 1개뿐입니다. 결과를 비교해보면 '더 넓게(neuron 추가)'와
'더 깊게(layer 추가)'가 같은 효과를 내지 않는다는 걸 알 수 있습니다 — 이 둘도 결국
hidden layer 개수 자체를 포함한 더 큰 hyperparameter tuning의 일부입니다.

## 이제 분류(classification) 문제로

지금까지는 숫자(정형 데이터)를 입력받아 숫자(대여량)를 예측하는 **회귀** 문제였습니다.
같은 PyTorch 도구(`nn.Module`, loss function, optimizer, 학습 루프)가 **이미지를
입력받아 클래스(0~9 숫자)를 예측하는 분류** 문제에도 그대로 적용된다는 걸 이번엔
**MNIST 손글씨 숫자** 데이터로 확인해봅니다.

In [ ]:
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

## 1. Load MNIST

`transforms.ToTensor()` converts each image to a tensor with pixel values
scaled to [0, 1].

In [ ]:
transform = transforms.ToTensor()

train_data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print(f"Training images: {len(train_data)}, test images: {len(test_data)}")

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray_r")
    ax.set_title(str(labels[i].item()))
    ax.axis("off")
plt.suptitle("A batch of MNIST digits")
plt.show()

## 2. Define the network

A simple feedforward network: flatten the 28x28 image into 784 numbers, pass
through one hidden layer with a ReLU activation, then a 10-way output layer
(one score per digit).

In [ ]:
def draw_layer(ax, x, n_shown, n_total, y_center=0, spacing=0.8, color="steelblue", r=0.3):
    half = n_shown // 2
    top_ys = [y_center + spacing * (half - i) + spacing * 1.1 for i in range(half)]
    bottom_ys = [y_center - spacing * (i + 1) - spacing * 1.1 for i in range(half)]
    ys = top_ys + bottom_ys
    for y in ys:
        ax.add_patch(plt.Circle((x, y), r, color=color, ec="white", linewidth=1.2, zorder=3))
    if n_total > n_shown:  # 너무 많은 노드는 가운데를 생략 부호(⋮)로 표시
        ax.text(x, y_center, "\u22ee", fontsize=20, ha="center", va="center", zorder=3)
    return ys


fig, ax = plt.subplots(figsize=(11, 8.5))
input_color, hidden_color, output_color = "#9e9e9e", "#7e6fd1", "#1f9c7a"

y0 = -1.5  # zoom-in 설명을 그릴 공간을 위쪽에 남겨두기 위해 전체 다이어그램을 아래로 이동
input_ys = draw_layer(ax, x=0, n_shown=4, n_total=784, y_center=y0, color=input_color)
hidden_ys = draw_layer(ax, x=4.5, n_shown=8, n_total=128, y_center=y0, color=hidden_color)
output_ys = draw_layer(ax, x=9, n_shown=10, n_total=10, y_center=y0, color=output_color, spacing=0.5)

for y1 in input_ys:
    for y2 in hidden_ys:
        ax.plot([0, 4.5], [y1, y2], color="gray", linewidth=0.4, alpha=0.45, zorder=1)
for y1 in hidden_ys:
    for y2 in output_ys:
        ax.plot([4.5, 9], [y1, y2], color="gray", linewidth=0.4, alpha=0.45, zorder=1)

bottom_y = min(input_ys + hidden_ys + output_ys) - 1.0
ax.text(0, bottom_y, "Input\n28x28 = 784", ha="center", fontsize=11, fontweight="bold")
ax.text(4.5, bottom_y, "Hidden (ReLU)\n128", ha="center", fontsize=11, fontweight="bold")
ax.text(9, bottom_y, "Output\n10", ha="center", fontsize=11, fontweight="bold")

# 은닉 노드 하나를 확대해서, 그 안에서 벌어지는 연산(weighted sum -> ReLU)을 보여줍니다.
zx, zy = 4.5, hidden_ys[0]
zoom_cx, zoom_cy, zoom_r = 4.5, 5.6, 2.3
ax.plot([zx, zoom_cx - zoom_r * 0.55], [zy, zoom_cy - zoom_r * 0.9], "--", color="black", linewidth=0.8, zorder=2)
ax.plot([zx, zoom_cx + zoom_r * 0.55], [zy, zoom_cy - zoom_r * 0.9], "--", color="black", linewidth=0.8, zorder=2)
ax.add_patch(plt.Circle((zoom_cx, zoom_cy), zoom_r, facecolor=hidden_color, edgecolor="black", linewidth=1.3, zorder=4))

ax.text(zoom_cx - 1.05, zoom_cy + 0.35, "$\\Sigma$", fontsize=34, color="white", ha="center", va="center", zorder=5)
ax.text(zoom_cx - 1.05, zoom_cy - 0.75, "weighted\nsum", fontsize=8, color="white", ha="center", va="center", zorder=5)
ax.annotate("", xy=(zoom_cx + 0.35, zoom_cy + 0.35), xytext=(zoom_cx - 0.45, zoom_cy + 0.35),
            arrowprops=dict(arrowstyle="->", color="white", lw=1.3), zorder=5)

relu_inset = ax.inset_axes([zoom_cx + 0.15, zoom_cy - 0.35, 1.7, 1.3], transform=ax.transData)
xs = np.linspace(-2, 2, 100)
relu_inset.plot(xs, np.maximum(0, xs), color="white", linewidth=1.8)
relu_inset.axhline(0, color="white", linewidth=0.5, alpha=0.6)
relu_inset.axvline(0, color="white", linewidth=0.5, alpha=0.6)
relu_inset.set_facecolor("none")
relu_inset.set_xticks([])
relu_inset.set_yticks([])
for spine in relu_inset.spines.values():
    spine.set_visible(False)
ax.text(zoom_cx + 1.05, zoom_cy - 0.75, "ReLU", fontsize=8, color="white", ha="center", va="center", zorder=5)

ax.set_xlim(-2, 11)
ax.set_ylim(-6.5, 9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("DigitClassifier: 784 -> 128 -> 10", fontsize=13)
plt.show()

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten (batch, 1, 28, 28) -> (batch, 784)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)             # raw scores ("logits"), one per digit
        return x

model = DigitClassifier().to(device)
print(model)
print("Total trainable parameters:", sum(p.numel() for p in model.parameters()))

## 3. The training loop

Every PyTorch training loop follows the same four steps for each batch:
1. **Forward pass** — run the batch through the model
2. **Loss** — how wrong were the predictions?
3. **Backward pass** — compute gradients of the loss w.r.t. every parameter
4. **Optimizer step** — nudge every parameter to reduce the loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

n_epochs = 3
history = {"train_loss": [], "test_acc": []}

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)          # 1. forward pass
        loss = criterion(outputs, labels)  # 2. loss
        loss.backward()                  # 3. backward pass (gradients)
        optimizer.step()                 # 4. optimizer step (update weights)

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_data)
    test_acc = evaluate(model, test_loader)
    history["train_loss"].append(train_loss)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch+1}/{n_epochs}  train_loss={train_loss:.4f}  test_acc={test_acc:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["test_acc"], marker="o", color="seagreen")
axes[1].set_title("Test Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Look at some mistakes

A trained model is rarely perfect — looking at *which* digits it confuses is
often more informative than the accuracy number alone.

In [ ]:
model.eval()
wrong_images, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images_d, labels_d = images.to(device), labels.to(device)
        preds = model(images_d).argmax(dim=1)
        mistakes = (preds != labels_d).cpu()
        if mistakes.any():
            wrong_images.extend(images[mistakes][:10 - len(wrong_images)])
            wrong_true.extend(labels[mistakes][:10 - len(wrong_true)])
            wrong_pred.extend(preds.cpu()[mistakes][:10 - len(wrong_pred)])
        if len(wrong_images) >= 10:
            break

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(wrong_images[i][0], cmap="gray_r")
    ax.set_title(f"true={wrong_true[i].item()}, pred={wrong_pred[i].item()}")
    ax.axis("off")
plt.suptitle("Misclassified Digits")
plt.tight_layout()
plt.show()